In [ ]:
!pip install rasterio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Import required libraries
import geopandas as gpd
import rasterio as rio
from rasterio.plot import show
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings

In [ ]:
# Read the point shapefiles for the whole studyarea.
file_path = '/content/drive/MyDrive/Flood_Susceptibility/Predict_Studyarea/studyarea_points04.shp'
points = gpd.read_file(file_path)
points.head()

,id,left,top,right,bottom,row_index,col_index,geometry
0,23.0,643240.765,1.128139e+06,643740.765,1.128639e+06,22.0,0.0,POINT (643240.765 1128139.336)
1,24.0,643240.765,1.127639e+06,643740.765,1.128139e+06,23.0,0.0,POINT (643240.765 1127639.336)
2,82.0,643740.765,1.128139e+06,644240.765,1.128639e+06,22.0,1.0,POINT (643740.765 1128139.336)
3,83.0,643740.765,1.127639e+06,644240.765,1.128139e+06,23.0,1.0,POINT (643740.765 1127639.336)
4,84.0,643740.765,1.127139e+06,644240.765,1.127639e+06,24.0,1.0,POINT (643740.765 1127139.336)


In [ ]:
# Remove columns 'pointid','grid_code' & 'geometry'
points = points.drop(columns=['id','left','top','bottom','right','row_index','col_index'])
points.head()

,geometry
0,POINT (643240.765 1128139.336)
1,POINT (643240.765 1127639.336)
2,POINT (643740.765 1128139.336)
3,POINT (643740.765 1127639.336)
4,POINT (643740.765 1127139.336)


In [ ]:
# Make columns to extract the values of each predictive feature.
points['Relief'] = np.nan  # Initialize with NaN for floating-point values
points['Slope'] = np.nan
points['Dis_River'] = np.nan
points['Drai_Density'] = np.nan
points['TWI'] = np.nan
points['LULC'] = np.nan
points['Geomorphology'] = np.nan
points['SPI'] = np.nan
points['Conv_Index'] = np.nan
points['Curvature'] = np.nan
points['TPI'] = np.nan
points['NDVI'] = np.nan
points['Rainfall'] = np.nan

points.head()

,geometry,Relief,Slope,Dis_River,Drai_Density,TWI,LULC,Geomorphology,SPI,Conv_Index,Curvature,TPI,NDVI,Rainfall
0,POINT (643240.765 1128139.336),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,POINT (643240.765 1127639.336),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,POINT (643740.765 1128139.336),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,POINT (643740.765 1127639.336),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,POINT (643740.765 1127139.336),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Raster filepaths
relief = '/content/drive/MyDrive/Flood_Susceptibility/DL_Data_Standardized/relief.tif'
slope = '/content/drive/MyDrive/Flood_Susceptibility/DL_Data_Standardized/slope.tif'
dis_river = '/content/drive/MyDrive/Flood_Susceptibility/DL_Data_Standardized/riverdist.tif'
drai_density = '/content/drive/MyDrive/Flood_Susceptibility/DL_Data_Standardized/dd.tif'
twi = '/content/drive/MyDrive/Flood_Susceptibility/DL_Data_Standardized/twi.tif'
lulc = '/content/drive/MyDrive/Flood_Susceptibility/DL_Data_Standardized/lulc.tif'
geomorph = '/content/drive/MyDrive/Flood_Susceptibility/DL_Data_Standardized/geomorphology.tif'
spi = '/content/drive/MyDrive/Flood_Susceptibility/DL_Data_Standardized/spi.tif'
conv_index = '/content/drive/MyDrive/Flood_Susceptibility/DL_Data_Standardized/ci.tif'
curvature = '/content/drive/MyDrive/Flood_Susceptibility/DL_Data_Standardized/curvature.tif'
tpi = '/content/drive/MyDrive/Flood_Susceptibility/DL_Data_Standardized/tpi.tif'
ndvi = '/content/drive/MyDrive/Flood_Susceptibility/DL_Data_Standardized/ndvi.tif'
rainfall = '/content/drive/MyDrive/Flood_Susceptibility/DL_Data_Standardized/rainfall.tif'

In [ ]:
# Open the predictive features (raster images) using rasterio.
relief_raster = rio.open(relief)
slope_raster = rio.open(slope)
dis_river_raster = rio.open(dis_river)
drai_density_raster = rio.open(drai_density)
twi_raster = rio.open(twi)
lulc_raster = rio.open(lulc)
geomorph_raster = rio.open(geomorph)
spi_raster = rio.open(spi)
conv_index_raster = rio.open(conv_index)
curvature_raster = rio.open(curvature)
tpi_raster = rio.open(tpi)
ndvi_raster = rio.open(ndvi)
rainfall_raster = rio.open(rainfall)

In [ ]:
# convert predictive Raster layers into Numpy array.
relief_arr = relief_raster.read(1)
slope_arr = slope_raster.read(1)
dis_river_arr = dis_river_raster.read(1)
drai_density_arr = drai_density_raster.read(1)
twi_arr = twi_raster.read(1)
lulc_arr = lulc_raster.read(1)
geomorph_arr = geomorph_raster.read(1)
spi_arr = spi_raster.read(1)
conv_index_arr = conv_index_raster.read(1)
curvature_arr = curvature_raster.read(1)
tpi_arr = tpi_raster.read(1)
ndvi_arr = ndvi_raster.read(1)
rainfall_arr = rainfall_raster.read(1)

In [ ]:
# Initialize columns with NaN to avoid dtype issues
warnings.filterwarnings("ignore", category=RuntimeWarning)
columns = ['Relief', 'Slope', 'Dis_River', 'Drai_Density', 'TWI', 'LULC',
           'Geomorphology', 'SPI', 'Conv_Index', 'Curvature', 'TPI', 'NDVI','Rainfall']
for col in columns:
    points[col] = np.nan

# Iterate over the rows to extract raster values
for index, row in points.iterrows():  # iterate over the rows in the shapefile.
    longitude = row['geometry'].x  # Get the longitude of the point.
    latitude = row['geometry'].y  # Get the latitude of the point.

    # Find the corresponding raster cell
    rowIndex, colIndex = relief_raster.index(longitude, latitude)

    # Assign values to columns explicitly using loc
    points.loc[index, 'Relief'] = relief_arr[rowIndex, colIndex]
    points.loc[index, 'Slope'] = slope_arr[rowIndex, colIndex]
    points.loc[index, 'Dis_River'] = dis_river_arr[rowIndex, colIndex]
    points.loc[index, 'Drai_Density'] = drai_density_arr[rowIndex, colIndex]
    points.loc[index, 'TWI'] = twi_arr[rowIndex, colIndex]
    points.loc[index, 'LULC'] = lulc_arr[rowIndex, colIndex]
    points.loc[index, 'Geomorphology'] = geomorph_arr[rowIndex, colIndex]
    points.loc[index, 'SPI'] = spi_arr[rowIndex, colIndex]
    points.loc[index, 'Conv_Index'] = conv_index_arr[rowIndex, colIndex]
    points.loc[index, 'Curvature'] = curvature_arr[rowIndex, colIndex]
    points.loc[index, 'TPI'] = tpi_arr[rowIndex, colIndex]
    points.loc[index, 'NDVI'] = ndvi_arr[rowIndex, colIndex]
    points.loc[index, 'Rainfall'] = rainfall_arr[rowIndex, colIndex]

# Check the updated GeoDataFrame
points.head()


,geometry,Relief,Slope,Dis_River,Drai_Density,TWI,LULC,Geomorphology,SPI,Conv_Index,Curvature,TPI,NDVI,Rainfall
0,POINT (643240.765 1128139.336),11.0,-9999.000000,1000.0,0.915152,6.431061,255.0,5.0,-3.402823e+38,9.838908,-0.111111,0.227273,0.000000,2330.183105
1,POINT (643240.765 1127639.336),10.0,-9999.000000,1000.0,0.899825,7.707060,255.0,5.0,-3.402823e+38,14.758362,0.111111,0.181818,0.000000,2348.015137
2,POINT (643740.765 1128139.336),10.0,1.012750,1000.0,0.913382,15.663495,1.0,5.0,5.067031e+00,-33.333332,-0.222222,-0.459459,0.743433,2310.079590
3,POINT (643740.765 1127639.336),8.0,3.199601,1000.0,0.897133,8.170497,0.0,5.0,4.567456e+00,-39.261166,-0.111111,-0.594595,0.771761,2326.180420
4,POINT (643740.765 1127139.336),9.0,0.675206,1000.0,0.877988,8.002930,3.0,5.0,2.870944e+00,-58.262466,-0.444444,-0.702703,0.680239,2341.572266


In [ ]:
print(points.isnull().sum())

geometry          0
Relief            0
Slope             0
Dis_River         0
Drai_Density      0
TWI               0
LULC              0
Geomorphology     0
SPI               0
Conv_Index        0
Curvature         0
TPI               0
NDVI             31
Rainfall          0
dtype: int64


In [ ]:
import pandas as pd
import numpy as np

# Define the valid ranges for each feature based on the images provided
valid_ranges = {
    'Relief': (3, 717),
    'Slope': (0, 58.9553),
    'Dis_River': (100, 1000),
    'Drai_Density': (0.0025849712, 4.8236695),
    'TWI': (3.3658473491669, 30.28600692749),
    'LULC': (1, 7),
    'Geomorphology': (1, 18),
    'SPI': (1.6694390773773, 15.068050384521),
    'Conv_Index': (-100, 100),
    'Curvature': (-10.111110687256, 7.4444446563721),
    'TPI': (-38.324325561523, 35.216217041016),
    'NDVI': (-0.4285714328289, 0.89060252904892),
    'Rainfall': (1849.0091552734, 3915.3872070312)  # Based on the new image provided
}

# Create a mask to identify points with values within valid ranges
mask = pd.Series(True, index=points.index)

# Check each column against its valid range
for column, (min_val, max_val) in valid_ranges.items():
    if column in points.columns:
        # Create a column-specific mask that's True for values within the valid range
        # Also consider NaN values as invalid
        column_mask = (
            (points[column] >= min_val) &
            (points[column] <= max_val) &
            (~points[column].isna()) &
            # Filter out extreme floating point values that might represent NoData
            (~np.isinf(points[column])) &
            (points[column] != -3.402823e+38) &
            (points[column] != 3.400000e+38) &
            (points[column] != -99999.0)
        )

        # Update the overall mask by performing an AND operation
        mask = mask & column_mask

# Apply the mask to filter the points
X_new = points[mask].copy()

# Check how many points were filtered out
print(f"Original number of points: {len(points)}")
print(f"Number of points after filtering: {len(X_new)}")
print(f"Number of points removed: {len(points) - len(X_new)}")

# Check the filtered GeoDataFrame
X_new.head()

# Optional: Save the filtered points to a new shapefile
# filtered_points.to_file("filtered_points.shp")

Original number of points: 2170
Number of points after filtering: 2064
Number of points removed: 106


,geometry,Relief,Slope,Dis_River,Drai_Density,TWI,LULC,Geomorphology,SPI,Conv_Index,Curvature,TPI,NDVI,Rainfall
2,POINT (643740.765 1128139.336),10.0,1.012750,1000.0,0.913382,15.663495,1.0,5.0,5.067031,-33.333332,-0.222222,-0.459459,0.743433,2310.079590
4,POINT (643740.765 1127139.336),9.0,0.675206,1000.0,0.877988,8.002930,3.0,5.0,2.870944,-58.262466,-0.444444,-0.702703,0.680239,2341.572266
5,POINT (643740.765 1126639.336),6.0,2.263636,1000.0,0.848125,17.728165,3.0,5.0,7.936322,-19.677816,-0.333333,-1.594595,0.846873,2363.924561
6,POINT (643740.765 1126139.336),9.0,0.477454,300.0,0.821285,15.102857,1.0,4.0,3.754438,100.000000,-0.000000,-0.189189,0.452289,2374.398682
7,POINT (643740.765 1125639.336),11.0,2.052775,200.0,0.765568,6.903668,3.0,5.0,3.972924,-5.120819,0.222222,0.379310,0.323103,2378.946289


In [ ]:
X_new.describe()

,Relief,Slope,Dis_River,Drai_Density,TWI,LULC,Geomorphology,SPI,Conv_Index,Curvature,TPI,NDVI,Rainfall
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,95.630329,8.514423,823.934109,1.037731,8.491820,2.699612,4.243217,6.237492,-1.144720,-0.010013,-0.016293,0.663352,2444.922161
std,140.578596,8.153347,314.427405,0.311731,3.838976,1.030735,2.728985,1.957675,30.817149,0.870024,4.005571,0.159874,380.997951
min,4.000000,0.337615,100.000000,0.007151,4.029218,1.000000,1.000000,1.669439,-100.000000,-5.222222,-18.594595,-0.029967,1850.104492
25%,13.000000,2.781886,1000.000000,0.991443,5.927149,3.000000,3.000000,4.795566,-17.620819,-0.444444,-2.114865,0.587944,2173.764587
50%,27.000000,5.520213,1000.000000,1.066957,6.956786,3.000000,4.000000,6.077108,0.000000,-0.000000,-0.135135,0.709153,2360.700195
75%,100.250000,11.204176,1000.000000,1.139763,9.574734,3.000000,5.000000,7.468712,15.948351,0.444444,1.945946,0.781072,2546.860291
max,690.000000,44.307690,1000.000000,2.866084,26.226385,5.000000,18.000000,13.852540,100.000000,5.222222,18.945946,0.882188,3877.000977


In [ ]:
# Save the points file
X_new.to_file('/content/drive/MyDrive/Flood_Susceptibility/Predict_Studyarea/Studyarea_point_filled02.shp') # save as a shapfile

/tmp/ipykernel_1003/3480801578.py:2: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  X_new.to_file('/content/drive/MyDrive/Flood_Susceptibility/Predict_Studyarea/Studyarea_point_filled02.shp') # save as a shapfile
